# Laboratorio Modelos de Clasificación — Ejercicio 1
## Santander Customer Satisfaction

**Versión con entrenamiento completo para Google Colab: compatible con “Entorno de ejecución → Ejecutar todo”.**

**Equipo:** Poma, Aldana, Brickzon, Vera y Marco. Los títulos y las celdas independientes de responsables permanecen en cada subsección.

**Qué hace Ejecutar todo:** instala dependencias faltantes, solicita `train.csv` cuando no esté disponible, prepara datos, define 4 escaladores + 6 clasificadores originales + 5 nuevos + 5 híbridos, evalúa **44 configuraciones individuales y 5 híbridos** con validación cruzada sobre entrenamiento, y finalmente ejecuta una evaluación única sobre holdout. Los resultados reales se construyen en tiempo de ejecución; no hay métricas prefabricadas.

**Importante:** la Parte A se conserva como *código histórico de consulta*, convertido en Markdown para no contaminar la Parte B. No se ejecuta. Las únicas celdas Python ejecutables corresponden a la Parte B.

El laboratorio original utiliza un dataset Santander que **no está incluido** en este notebook. Al pulsar Ejecutar todo se abrirá un selector de archivos en Google Colab si falta `train.csv`. Puedes cargar los tres CSV o usar Drive según la primera celda ejecutable.

## Guía de ejecución integral en Google Colab

1. Abrir este archivo `.ipynb` con Google Colab y escoger **Entorno de ejecución → Ejecutar todo**.
2. Si Colab solicita archivos, seleccionar `train.csv`, `test.csv` y `sample_submission.csv` de Santander. **Solo `train.csv` es indispensable para entrenamiento y evaluación**; los otros dos se conservan como archivos Kaggle no etiquetados. Si están en `/content` o en una ruta de `SANTANDER_DATA_DIR`, no habrá solicitud.
3. La celda 4.1.1 instala paquetes faltantes en el entorno e identifica los archivos. No ejecuta la Parte A.
4. **Modo predeterminado `completo`:** compara las 49 configuraciones usando todas las filas de entrenamiento; el holdout permanece separado. Esto exige más memoria y procesamiento en Colab.
5. Si Colab no tiene recursos suficientes, cambiar `MAX_FILAS_ENTRENAMIENTO = None` a `8000` en 4.1.1. Entonces la ejecución conserva todos los modelos, pero utiliza una muestra estratificada de entrenamiento.
6. El notebook ejecuta una sola vez la validación y el holdout; los escaladores se ajustan dentro de cada fold y los datos Kaggle sin `TARGET` nunca se usan como conjunto de métricas.
7. Si alguna configuración falla, el flujo final se detendrá antes del holdout e indicará el error. **No se inventan ni omiten resultados silenciosamente.**

**Orden automático:** configuración → división estratificada → pipelines → registros de modelos → evaluación CV → selección → holdout → tablas/gráficas/interpretación.

# Parte A — Notebook original del docente (referencia no ejecutable)

Las celdas originales de Santander se mantienen a continuación **como bloques de código en Markdown**, para consulta y comparación pedagógica. No se ejecutan con «Ejecutar todo», porque incluyen remuestreo y escalamiento antes de la división, además de una implementación incorrecta de ROC-AUC y configuraciones GPU que pueden fallar en Colab.

La **Parte B** contiene el flujo integrado y corregido que se utiliza para este laboratorio.

## Importar librerias

Desde equipos de soporte de primera línea hasta C-suites, la satisfacción del cliente es una medida clave del éxito. Los clientes insatisfechos no se quedan. Además, los clientes descontentos rara vez expresan su insatisfacción antes de irse.

El objetivo es identificar a los clientes insatisfechos en una etapa temprana de su relación. Hacerlo le permitiría a Santander tomar medidas proactivas para mejorar la felicidad de un cliente antes de que sea demasiado tarde.

En esta competencia, trabajará con cientos de funciones anónimas para predecir si un cliente está satisfecho o insatisfecho con su experiencia bancaria.

Se le proporciona un conjunto de datos anonimizados que contiene una gran cantidad de variables numéricas. La columna "OBJETIVO" es la variable a predecir. Es igual a uno para clientes insatisfechos y 0 para clientes satisfechos.

La tarea es predecir la probabilidad de que cada cliente en el conjunto de prueba sea un cliente insatisfecho.
Descripciones de archivos

1. train.csv: el conjunto de entrenamiento que incluye el objetivo  --- 76020
2. test.csv: el conjunto de prueba sin el objetivo  --- 75818
3. sample_submission.csv: un archivo de envío de muestra en el formato correcto

**Código original del docente (solo referencia; no se ejecuta):**

```python
#generales
import numpy as np
import pandas as pd

#gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#procesamiento
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, RandomizedSearchCV

#modelo
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB

#evaluación
from scipy import stats
from sklearn.metrics import accuracy_score, roc_curve, auc, mean_squared_error, f1_score

import warnings
warnings.filterwarnings('ignore')
```

* Si no cuenta con alguna de las librerias ejecute el seguiente codigo según corresponda

**Instalación mencionada en la plantilla original (referencia, no ejecutar literalmente):**

```bash
pip install catboost
pip install xgboost
pip install lightgbm
pip install seaborn
pip install scipy
```

## Carga de datos

**Código original del docente (solo referencia; no se ejecuta):**

```python
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")
```

## 1. Análisis Exploratorio

**1.1. muestreo aleatorio del conjunto de datos**

**Código original del docente (solo referencia; no se ejecuta):**

```python
train.sample(5)
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
test.sample(5)
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
sample.sample(5)
```

**1.2. Dimención de la estructura**

**Código original del docente (solo referencia; no se ejecuta):**

```python
#determinar la dimensionalidad de los conjuntos de datos
# shape (instancias, concepto)
print("Shape de train dataframe : {}".format(train.shape))
print("Shape de test dataframe : {}".format(test.shape))
print("Shape de sample dataframe : {}".format(sample.shape))
```

**1.3. Descripción y estadística basica**

**Código original del docente (solo referencia; no se ejecuta):**

```python
#Comprobar la existencia de null's en los conjuntos de datos
print('Null en train : {}'.format(train.isnull().sum().any()))
print('Null en test : {}'.format(test.isnull().sum().any()))
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
#análisis estadistico básico de las columnas numericas
train.describe()
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
#tipos de datos
train.info()
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
#tipos de datos
test.info()
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
# Determinar la existencia de duplicados
dupli = train[train.duplicated()]
dupli
```

**1.4. Análisis Gráfico**

**Código original del docente (solo referencia; no se ejecuta):**

```python
plt.figure(figsize = (20,20))
for i in range (50):
    plt.subplot(5, 10, i+1)
    sns.boxplot(train.iloc[:,i])
    plt.xlabel(train.columns[i], size = 10)
```

### 2. Modificación

**Código original del docente (solo referencia; no se ejecuta):**

```python
# remover columnas con valores unicos
for i in train.columns:
    if train[i].nunique() == 1:
        print(i)
        train.drop(i, inplace = True, axis = 1)
        test.drop(i, inplace = True, axis = 1)
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
def Outliers(data, ft):
    IQ1 = data[ft].quantile(0.25)
    IQ3 = data[ft].quantile(0.75)
    IQR = IQ3 - IQ1
    
    lower_bound = IQ1 - 1.5 * IQR
    upper_bound = IQ3 + 1.5 * IQR
    
    return lower_bound, upper_bound
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
def replace_outliers_with_iqr(train):
    # Recorremos cada columna del DataFrame
    
    for i in train.columns:
        if i != 'TARGET':
            # Solo aplicar a columnas numéricas
            if train[i].dtype in ['int64', 'float64']:
                # Obtener los límites de los outliers
                lower_bound, upper_bound = Outliers(train, i)
                
                # Reemplazar valores que están por debajo del límite inferior
                train.loc[train[i] < lower_bound, i] = lower_bound
                
                # Reemplazar valores que están por encima del límite superior
                train.loc[train[i] > upper_bound, i] = upper_bound
            
    return train
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
train = replace_outliers_with_iqr(train)
train
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
print(train['TARGET'].value_counts())
plt.figure(figsize = (8,5))
sns.countplot(x = train['TARGET'])
plt.xlabel('Target', size = 12)
plt.ylabel('Count', size = 12)
plt.title('Distribución en la columna de target antes del remuestreo', size = 12)
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
#eliminación de columnas 
test_id = test['ID']
train.drop('ID', inplace = True, axis = 1)
test.drop('ID', inplace = True, axis = 1)
```

**Separación de variables entrada y target**

**Código original del docente (solo referencia; no se ejecuta):**

```python
x = train.drop('TARGET', axis = 1)
x.head()
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
y = train.loc[:, 'TARGET']
y.head()
```

**Técnicas de muestreo**

**Código original del docente (solo referencia; no se ejecuta):**

```python
# Muestreo de los datos, para equilibrar las clases en el target.
sampler = RandomUnderSampler(random_state=123, sampling_strategy=.5)
x, y = sampler.fit_resample(x, y)
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
print(y.value_counts())
plt.figure(figsize = (8,5))
sns.countplot(x = y)
plt.xlabel('Target', size = 12)
plt.ylabel('Count', size = 12)
plt.title('Distribución en la columna de target despues del remuestreo', size = 12)
```

**Código original del docente (solo referencia; no se ejecuta):**

```python
# aplicar técnicas de escalado
# máximos minimos y standar

col = x.columns
std = StandardScaler()
x_std = std.fit_transform(x)
x_std = pd.DataFrame(data = x_std, columns = col)

mms = MinMaxScaler()
x_mms = mms.fit_transform(x)
x_mms = pd.DataFrame(data = x_mms, columns = col)
```

### 3. test y entrenamiento

**Código original del docente (solo referencia; no se ejecuta):**

```python
x_mms.head()
```

**Creación de clase model**

**Código original del docente (solo referencia; no se ejecuta):**

```python
def Models(model, name, d, xtrain, ytrain, xtest, ytest):
    print("Working on {} model".format(name))
    
    cla = model
    cla.fit(xtrain, ytrain)
    
    predicted = cla.predict(xtrain)
    tr_auc = accuracy_score(predicted, ytrain)*100
    
    predicted = cla.predict(xtest)
    te_auc = accuracy_score(predicted, ytest)*100
    
    F1 = f1_score(predicted, ytest)
    fpr, tpr, threshould = roc_curve(predicted, ytest)
    AUC = auc(fpr, tpr)
    
    d['Name'].append(name)
    d['Training ACU'].append(tr_auc)
    d['Testing ACU'].append(te_auc)
    d['F1'].append(F1)
    d['AUC'].append(AUC)
    
    print("**********"*5)
    print()
    return d, cla
```

**Evaluación de modelos segú el conjunto**

**Código original del docente (solo referencia; no se ejecuta):**

```python
l = [[x, 'Normal'], [x_std, 'Standard'], [x_mms, 'Min Max']]
final = []
model_fit = []
for i in l:
    xtrain, xtest, ytrain, ytest = train_test_split(i[0], y, test_size = 0.25, random_state = 42, stratify = y)
    
    d = {'Name' : [], 'Training ACU': [], 'Testing ACU': [], 'F1': [], 'AUC': []}
    
    models = [ 
        [RandomForestClassifier(n_estimators = 200), 'Random Forest'], 
        [DecisionTreeClassifier(), 'Decision Tree'], 
        [XGBClassifier(tree_method='gpu_hist'), 'XGBoost'], 
        [CatBoostClassifier(task_type="GPU"), 'CatBoost'],
        [GaussianNB(), 'Naive Bayes'], 
        [LogisticRegression(), 'Logistic Regression'] 
    ]

    for model in models:
        d, m = Models(model[0], model[1], d, xtrain, ytrain, xtest, ytest)
        model_fit.append([m, model[1], i[1]])
    final.append(d)
```

**Evaluación de los resultados por modelo y tipo de conjunto de datos**

**Código original del docente (solo referencia; no se ejecuta):**

```python
name = ['Normal', 'Standard', 'Min Max']
for i in range (len(name)):
    print(name[i])
    acu_data = pd.DataFrame(data = final[i])
    print(acu_data)
    print("******"*12)
```

# EJERCICIO

* Emplear cuatro modelos de escalamiento; por ejemplo:
    - robust Scaler
    - Basado en modelos (knn, etc.)

* agregar cinco modelos; por ejemplo:
    - SVM
    - AdaBoost

* preparar cinco modelos hibridos en base a los modelos previamente vistos; por ejemplo aplicando la libreria:
    - VotingClassifier

# Parte B — Desarrollo grupal integrado y ejecutable

**Todos los bloques están implementados.** «Ejecutar todo» ejecutará las configuraciones, entrenamientos y gráficas del laboratorio mediante la celda 4.6.5, al final. No se requiere lanzar celdas manualmente entre subsecciones.

**Contrato de integración (no renombrar):** `X_train`, `X_test`, `y_train`, `y_test` son las particiones etiquetadas; `escaladores`, `modelos_originales`, `modelos_nuevos`, `modelos_hibridos` son diccionarios con 4, 6, 5 y 5 entradas. `crear_pipeline_santander`, `evaluar_modelo`, `ejecutar_comparacion`, `evaluar_holdout_final` gestionan transformaciones y evaluación. `tabla_validacion_ultima` contiene CV; `tabla_resultados` contiene holdout; `tabla_hibridos` resume los cinco ensambles.

**Protocolo:** 75 % entrenamiento / 25 % holdout estratificado, `random_state=42`, aprendizaje del preprocesamiento solo en folds de entrenamiento y *holdout único*. Por defecto se usan todas las filas de entrenamiento; el holdout se conserva separado. Se muestran los tamaños efectivos.

**Responsabilidades originales:** Poma (4.1.2–4.1.3, 4.2), Aldana (4.4), Brickzon (4.5.1–4.5.3), Vera (4.5.4–4.5.5, 4.6.2), Marco (4.1.1, 4.1.4, 4.3, 4.6.1 y 4.6.3–4.6.5).

## 4.1. Preparación e interfaz compartida

Este bloque crea los objetos que usarán todas las secciones. Poma y Marco deben cerrar el contrato de nombres antes de que Aldana, Brickzon y Vera empiecen a integrar. Todas las celdas de implementación siguen vacías.

### 4.1.1. Preparar entorno Colab y localizar archivos

> **Responsable: Marco**

**Implementar:** instalar/importar las bibliotecas necesarias en Colab; elegir un método de acceso a datos (Google Drive o subida manual); declarar `DATA_DIR` con una ruta **real**; comprobar si existen `train.csv`, `test.csv` y `sample_submission.csv` antes de leerlos. No incrustar credenciales.

**Sugerencias de comandos (solo como guía):** `%pip install -q catboost xgboost lightgbm imbalanced-learn`. Para Drive: `from google.colab import drive` y `drive.mount('/content/drive')`. Para subida puntual: `from google.colab import files` y `files.upload()`.

**Salida esperada:** mostrar las versiones de Python y `scikit-learn`, la ruta seleccionada y la existencia de los tres CSV. Si faltan, informar claramente cuál falta.

**Verificación:** ejecutar sin rutas de Windows ni referencias a directorios locales ajenos al entorno Colab.

In [ ]:
# 4.1.1 — Configuración única para Google Colab (Marco)
from pathlib import Path
import os, sys, subprocess, importlib.util, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import sklearn
from importlib.metadata import version as _pkg_version, PackageNotFoundError
from packaging.version import Version

RANDOM_STATE = 42
MAX_FILAS_ENTRENAMIENTO = None  # Dataset completo; cambiar a 8000 si Colab agota recursos
CV_SPLITS = 2                  # CV externa; 3 folds internos en Stacking
STRICT_ERRORS = True          # bloquear holdout si una configuración falla
np.random.seed(RANDOM_STATE)

# Instalar paquetes faltantes y CatBoost compatible con el API de tags de sklearn 1.8+.
# CatBoost <=1.2.8 produce AttributeError __sklearn_tags__ con sklearn 1.8.
for modulo, paquete in [('xgboost', 'xgboost'), ('lightgbm', 'lightgbm'), ('catboost', 'catboost')]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if Version(sklearn.__version__) >= Version('1.8'):
    try:
        version_catboost = Version(_pkg_version('catboost'))
    except PackageNotFoundError:
        version_catboost = Version('0')
    if version_catboost < Version('1.2.9'):
        print('Actualizando CatBoost >=1.2.9 para compatibilidad con scikit-learn', sklearn.__version__)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'catboost>=1.2.9'])

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, VotingClassifier, StackingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Si has montado Drive anteriormente, configura SANTANDER_DATA_DIR en Colab.
# De no existir train.csv en las rutas comunes, aparece un único selector de archivos.
base_env = os.environ.get('SANTANDER_DATA_DIR')
candidatas = [Path(base_env)] if base_env else []
candidatas += [Path('/content'), Path.cwd(), Path('/content/drive/MyDrive/Santander')]
DATA_DIR = next((p for p in candidatas if (p / 'train.csv').is_file()), None)
if DATA_DIR is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'No encuentro train.csv. En Colab, súbelo a /content o configura SANTANDER_DATA_DIR.'
        ) from exc
    print('Falta train.csv: selecciona los CSV del ejercicio Santander en el cuadro de Colab.')
    recibidos = files.upload()
    if 'train.csv' not in recibidos and not Path('/content/train.csv').is_file():
        raise FileNotFoundError('El archivo train.csv de Santander es obligatorio.')
    DATA_DIR = Path('/content')
RUTAS_CSV = {nombre: DATA_DIR / nombre for nombre in
             ('train.csv', 'test.csv', 'sample_submission.csv')}
if not RUTAS_CSV['train.csv'].is_file():
    raise FileNotFoundError(f'No existe {RUTAS_CSV["train.csv"]}')
print('Python:', sys.version.split()[0], '| scikit-learn:', sklearn.__version__)
print('Dataset:', DATA_DIR.resolve(), '| CSV presentes:',
      {nombre: p.is_file() for nombre,p in RUTAS_CSV.items()})
print('Modo:', 'COMPLETO' if MAX_FILAS_ENTRENAMIENTO is None else 'INTEGRAL ACOTADO',
      '| CV folds:', CV_SPLITS)


### 4.1.2. Cargar el dataset y construir la división estratificada

> **Responsable: Poma**

**Implementar:** recargar **desde los archivos originales**, sin reutilizar `train`, `x` ni `y` manipulados en la Parte A. Separar `TARGET`, retirar `ID` de los predictores, conservar `ID` aparte si luego se generan predicciones para Kaggle. Construir `X_train`, `X_test`, `y_train`, `y_test` con `train_test_split(test_size=0.25, stratify=y, random_state=42)`.

**Reglas:** `test.csv` carece de `TARGET`; **no** usarlo para métricas. No aplicar ADASYN, RandomUnderSampler, SMOTE, escaladores, clipping de valores atípicos ni selección basada en estadísticas del dataset completo **antes del split**.

**Salida esperada:** dimensiones, lista de características numéricas/no numéricas, distribución de `TARGET` antes y después del split, y confirmación de que los conjuntos no comparten registros.

**Verificación:** mismas columnas y orden para `X_train`/`X_test`; valores binarios esperados 0 y 1, proporciones estratificadas similares.

In [ ]:
# 4.1.2 — Datos y partición 75/25 sin fuga (Poma)
train_original = pd.read_csv(RUTAS_CSV['train.csv'], low_memory=False)
requeridas = {'ID', 'TARGET'}
if not requeridas.issubset(train_original.columns):
    raise ValueError(f'Faltan columnas Santander: {sorted(requeridas - set(train_original))}')
if train_original['ID'].isna().any() or train_original['ID'].duplicated().any():
    raise ValueError('ID debe ser único y no tener valores ausentes.')
if train_original['TARGET'].isna().any() or set(train_original['TARGET'].unique()) != {0,1}:
    raise ValueError('TARGET debe contener solamente las dos clases 0 y 1, sin nulos.')
y_completo = train_original['TARGET'].astype('int8')
X_completo = train_original.drop(columns=['ID','TARGET'])
if len(X_completo.select_dtypes(exclude='number').columns):
    raise TypeError('Hay columnas no numéricas no previstas en Santander.')
X_train_total, X_test, y_train_total, y_test = train_test_split(
    X_completo, y_completo, test_size=.25, stratify=y_completo,
    random_state=RANDOM_STATE
)
if MAX_FILAS_ENTRENAMIENTO is not None and len(X_train_total) > MAX_FILAS_ENTRENAMIENTO:
    X_train, _, y_train, _ = train_test_split(
        X_train_total, y_train_total, train_size=int(MAX_FILAS_ENTRENAMIENTO),
        stratify=y_train_total, random_state=RANDOM_STATE
    )
else:
    X_train, y_train = X_train_total.copy(), y_train_total.copy()
if len(X_train.index.intersection(X_test.index)):
    raise AssertionError('Particiones con índices compartidos.')
if not X_train.columns.equals(X_test.columns):
    raise AssertionError('Diferente orden de columnas entre train y holdout.')
if min(y_train.value_counts()) < 3 * CV_SPLITS:
    raise ValueError('La muestra elegida contiene muy pocos ejemplos de una clase para la CV interna.')
print('Filas originales:', len(X_completo), '| entrenamiento disponible:', len(X_train_total),
      '| entrenamiento efectivo:', len(X_train), '| holdout:', len(X_test))
print('Proporciones TARGET:', y_train.value_counts(normalize=True).sort_index().round(4).to_dict(),
      '| holdout:', y_test.value_counts(normalize=True).sort_index().round(4).to_dict())
print('Predictores:', X_train.shape[1], '| no se usa ID ni el test.csv de Kaggle para evaluar')


### 4.1.3. Definir preprocesamiento reproducible y libre de fuga de datos

> **Responsable: Poma**

**Implementar:** proponer un `Pipeline` o una función de construcción de pipelines que ajuste **en entrenamiento** la imputación de faltantes, el escalamiento seleccionado y el clasificador. Si se eliminan columnas constantes o se limitan outliers, aprender esas reglas solo con entrenamiento y aplicar la misma transformación a holdout. Elegir y documentar una estrategia uniforme de desbalanceo (por defecto: sin remuestreo; si usan muestreo, hacerlo con un `imblearn.pipeline.Pipeline` dentro del entrenamiento y de cada fold).

**No hacer:** transformar `X_test` con un `fit_transform`, ni aplicar undersampling/SMOTE a los datos de prueba. No usar `ID` como predictor.

**Salida esperada:** objeto/fábrica reutilizable en las secciones 4.3–4.5; una breve nota de las transformaciones y su orden.

**Verificación:** `fit` ocurre solo sobre muestras de entrenamiento; transformaciones idénticas en las demás particiones.

In [ ]:
# 4.1.3 — Transformaciones que se ajustan solo durante fit (Poma)
class EliminarColumnasVacias(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        frame = pd.DataFrame(X)
        self.mascara_ = ~frame.isna().all(axis=0).to_numpy()
        if not self.mascara_.any():
            raise ValueError('Todas las columnas carecen de datos en el entrenamiento del fold.')
        self.columnas_ = list(frame.columns[self.mascara_])
        return self
    def transform(self, X):
        frame = pd.DataFrame(X)
        if frame.shape[1] != len(self.mascara_):
            raise ValueError('Número de columnas diferente del utilizado en fit.')
        return frame.iloc[:, self.mascara_]

def crear_pipeline_santander(clasificador, escalador=None):
    """Fábrica común; Stacking recibe preprocesamiento dentro de cada rama."""
    if isinstance(clasificador, StackingClassifier):
        if escalador is not None:
            raise ValueError('El escalamiento de Stacking pertenece a sus ramas internas.')
        for nombre, estimador in clasificador.estimators:
            if not isinstance(estimador, Pipeline):
                raise TypeError(f'La rama {nombre} de Stacking debe ser un Pipeline completo.')
        return Pipeline([('clasificador', clone(clasificador))])
    pasos = [
        ('eliminar_vacias', EliminarColumnasVacias()),
        ('imputador', SimpleImputer(strategy='median')),
        ('selector_constantes', VarianceThreshold(threshold=0.0)),
    ]
    if escalador is not None:
        pasos.append(('escalador', clone(escalador)))
    pasos.append(('clasificador', clone(clasificador)))
    return Pipeline(pasos)

def demostrar_escalador_en_train(escalador):
    columnas = [c for c in X_train.columns if X_train[c].nunique(dropna=True)>1][:8]
    if not columnas: raise ValueError('No hay columnas variables para demostración.')
    prep = Pipeline([('vacias', EliminarColumnasVacias()),
        ('imputador', SimpleImputer(strategy='median')),
        ('constantes', VarianceThreshold())])
    sample = prep.fit_transform(X_train[columnas])
    return clone(escalador).fit_transform(sample), columnas
print('Fábrica de pipelines disponible, sin transformaciones ajustadas sobre el holdout.')


### 4.1.4. Definir la interfaz de evaluación y resultados

> **Responsable: Marco**

**Implementar:** una función `evaluar_modelo(...)` (o interfaz equivalente, documentando argumentos) que reciba nombre, tipo (`original`, `nuevo` o `hibrido`), clasificador y escalador; construya el pipeline adecuado, entrene en `X_train/y_train` y calcule métricas sobre validación/holdout según el protocolo. Registrar tiempo de entrenamiento y resultados en una **estructura común**, sin cambiar su forma entre integrantes.

**Columnas obligatorias de la tabla final:** `tipo`, `modelo`, `escalador`, `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `tiempo_s`. Agregar `etapa` (`validacion`/`holdout`) y `semilla` si conviene.

**ROC-AUC correcto:** usar `roc_auc_score(y_real, proba_clase_1)` o `decision_function`; `roc_curve(y_real, puntaje_clase_1)` para curvas ROC. **No** usar `roc_curve(y_pred, y_real)` como en la función original. Controlar divisiones por cero al calcular Precision/Recall/F1. Nunca decidir el mejor modelo basándose reiteradamente en el holdout.

**Validación:** prueba de la interfaz con un clasificador sencillo sobre una partición de validación interna sin modificar el test final; conservar las salidas compatibles con los tres diccionarios de modelos.

In [ ]:
# 4.1.4 — Interfaz común de evaluación (Marco)
COLUMNAS_RESULTADOS = ['tipo','modelo','escalador','accuracy','precision','recall','f1',
                       'roc_auc','tiempo_s','etapa','fold','matriz_confusion']
resultados = []
tabla_resultados = pd.DataFrame(columns=COLUMNAS_RESULTADOS)
tabla_validacion_ultima = pd.DataFrame()
modelos_holdout = {}
predicciones_holdout = {}

def _puntaje_continuo(modelo, X):
    """Puntuación para TARGET=1; no sustituye el score por etiquetas duras."""
    classes = list(modelo.classes_)
    if 1 not in classes: return None
    if hasattr(modelo, 'predict_proba'):
        prob = np.asarray(modelo.predict_proba(X))
        return prob[:, classes.index(1)]
    if hasattr(modelo, 'decision_function'):
        valor = np.asarray(modelo.decision_function(X))
        if valor.ndim == 1:
            return valor if classes[-1] == 1 else -valor
        return valor[:,classes.index(1)]
    return None

def evaluar_modelo(nombre, tipo, estimador, escalador, X_fit, y_fit, X_eval, y_eval,
                   etapa='validacion', fold=None, guardar_ajustado=False):
    if len(X_fit.index.intersection(X_eval.index)):
        raise ValueError('Hay filas compartidas entre ajuste y evaluación.')
    if etapa == 'validacion':
        if not X_fit.index.isin(X_train.index).all() or not X_eval.index.isin(X_train.index).all():
            raise ValueError('La validación solo puede usar filas procedentes de X_train.')
    elif etapa == 'holdout':
        if not X_fit.index.equals(X_train.index) or not X_eval.index.equals(X_test.index):
            raise ValueError('Holdout debe utilizar X_train y X_test completos y en orden.')
    else:
        raise ValueError('Etapa desconocida.')
    modelo = crear_pipeline_santander(estimador, escalador)
    inicio = time.perf_counter()
    modelo.fit(X_fit, y_fit)
    elapsed = time.perf_counter() - inicio
    y_real, pred = np.asarray(y_eval), np.asarray(modelo.predict(X_eval))
    score = _puntaje_continuo(modelo, X_eval)
    auc = float(roc_auc_score(y_real, score)) if score is not None and len(set(y_real)) == 2 else np.nan
    fila = dict(tipo=tipo, modelo=nombre,
        escalador='sin_escalado' if escalador is None else next(
            (k for k,v in escaladores.items() if v is escalador), type(escalador).__name__),
        accuracy=float(accuracy_score(y_real, pred)),
        precision=float(precision_score(y_real, pred, zero_division=0)),
        recall=float(recall_score(y_real, pred, zero_division=0)),
        f1=float(f1_score(y_real, pred, zero_division=0)), roc_auc=auc,
        tiempo_s=float(elapsed), etapa=etapa, fold=fold,
        matriz_confusion=confusion_matrix(y_real, pred, labels=[0,1]).tolist() if etapa=='holdout' else None)
    if guardar_ajustado and etapa=='holdout':
        modelos_holdout[(tipo,nombre)] = modelo
        predicciones_holdout[(tipo,nombre)] = {'y_true': y_real, 'score':score, 'y_pred':pred}
    return fila


## 4.2. Cuatro métodos de escalamiento

Crear el diccionario `escaladores` con **exactamente** estas cuatro entradas: `standard`, `minmax`, `robust`, `quantile`. **No ajustar** los transformadores al crearlos: dejarlos listos para integrarse en los pipelines y ajustar con `fit` exclusivamente sobre el entrenamiento. Comparar qué cambia en distribución/rangos sin usar información del holdout.

### 4.2.1. StandardScaler

> **Responsable: Poma**

**Implementar:** incorporar `StandardScaler` al diccionario `escaladores` con la clave `standard`. Explicar en Markdown la estandarización mediante media y desviación típica.

**Verificación:** comprobar en un subconjunto **de entrenamiento transformado** media aproximadamente 0 y desviación aproximadamente 1 para columnas no constantes; no ajustar al holdout.

**Entregable:** transformador registrable y una interpretación breve.

In [ ]:
# 4.2.1 — StandardScaler (Poma)
escaladores = {'standard': StandardScaler()}
demo, cols = demostrar_escalador_en_train(escaladores['standard'])
print('StandardScaler: media', np.round(demo.mean(axis=0), 3).tolist(),
      '| desviación', np.round(demo.std(axis=0), 3).tolist())


### 4.2.2. MinMaxScaler

> **Responsable: Poma**

**Implementar:** incorporar `MinMaxScaler` con clave `minmax`, rango de entrenamiento `[0, 1]` (a menos que documenten otra elección).

**Verificación:** revisar los mínimos y máximos del **entrenamiento transformado**; valores del holdout pueden quedar fuera de `[0,1]` si exceden el rango aprendido, y eso no debe corregirse con un reajuste.

**Entregable:** transformador y breve interpretación.

In [ ]:
# 4.2.2 — MinMaxScaler (Poma)
escaladores['minmax'] = MinMaxScaler()
demo, cols = demostrar_escalador_en_train(escaladores['minmax'])
print('MinMaxScaler: mínimos', np.round(demo.min(axis=0), 3).tolist(),
      '| máximos', np.round(demo.max(axis=0), 3).tolist())


### 4.2.3. RobustScaler

> **Responsable: Poma**

**Implementar:** incorporar `RobustScaler` con clave `robust`; explicar por qué mediana y rango intercuartílico reducen la influencia de valores extremos.

**Verificación:** comparar con StandardScaler usando ejemplos de características con outliers del **entrenamiento**. No eliminar automáticamente valores atípicos sin justificarlo.

**Entregable:** transformador y breve interpretación.

In [ ]:
# 4.2.3 — RobustScaler (Poma)
escaladores['robust'] = RobustScaler()
demo, cols = demostrar_escalador_en_train(escaladores['robust'])
print('RobustScaler: medianas', np.round(np.median(demo, axis=0), 3).tolist())


### 4.2.4. QuantileTransformer

> **Responsable: Poma**

**Implementar:** incorporar `QuantileTransformer` con clave `quantile`, `output_distribution='normal'`, `random_state=42` y un `n_quantiles` no mayor que el tamaño de entrenamiento usado para ajustar.

**Verificación:** confirmar transformación de distribuciones en una muestra de entrenamiento; explicar que este método puede cambiar distancias/relaciones y por eso debe compararse empíricamente.

**Entregable:** cuarto transformador; mostrar `list(escaladores)` con **exactamente cuatro** claves distintas.

In [ ]:
# 4.2.4 — QuantileTransformer (Poma)
escaladores['quantile'] = QuantileTransformer(n_quantiles=min(1000, len(X_train)),
    output_distribution='normal',random_state=RANDOM_STATE)
demo, cols = demostrar_escalador_en_train(escaladores['quantile'])
assert set(escaladores)=={'standard','minmax','robust','quantile'}
print('QuantileTransformer: media ejemplo', np.round(demo.mean(axis=0),3).tolist(),
      '| métodos:', list(escaladores))


## 4.3. Seis clasificadores originales como línea base

### 4.3.1. Reconstruir los seis modelos de la clase bajo el nuevo protocolo

> **Responsable: Marco**

**Implementar:** registrar en `modelos_originales` estos seis clasificadores utilizados en el notebook Santander del docente: `random_forest`, `decision_tree`, `xgboost`, `catboost`, `gaussian_nb`, `logistic_regression`. Usar semillas cuando corresponda y una configuración razonable para Colab.

**Precaución:** el notebook original fuerza GPU en XGBoost/CatBoost y podría fallar en el entorno actual. Para esta plantilla, preferir configuraciones CPU compatibles; solo activar GPU cuando se haya verificado disponibilidad. Evitar copiar la función original `Models()` por su implementación incorrecta de ROC.

**Comparación:** entrenar y evaluar las líneas base con la interfaz común, sin mezclar métricas del notebook original con las de esta Parte B. Para cuantificar el efecto del escalamiento, usar los cuatro transformadores en condiciones comparables; si el costo impide evaluar algunas combinaciones, justificar explícitamente qué se ejecutó y qué no.

**Salida esperada:** `len(modelos_originales) == 6` y resultados trazables con el mismo split/semilla/métricas.

In [ ]:
# 4.3.1 — Seis clasificadores originales (Marco)
# Presupuestos de árboles moderados para que la matriz 44 + 5 sea viable en Colab.
modelos_originales = {
    'random_forest':RandomForestClassifier(n_estimators=90, min_samples_leaf=2,
        class_weight='balanced',n_jobs=2,random_state=RANDOM_STATE),
    'decision_tree':DecisionTreeClassifier(max_depth=12,min_samples_leaf=10,
        class_weight='balanced',random_state=RANDOM_STATE),
    'xgboost':XGBClassifier(n_estimators=90,max_depth=5,learning_rate=.06,subsample=.8,
        colsample_bytree=.8,objective='binary:logistic',eval_metric='logloss',
        tree_method='hist',n_jobs=2,random_state=RANDOM_STATE),
    'catboost':CatBoostClassifier(iterations=90,depth=6,learning_rate=.06,
        loss_function='Logloss',verbose=False,allow_writing_files=False,
        thread_count=2,random_seed=RANDOM_STATE),
    'gaussian_nb':GaussianNB(),
    'logistic_regression':LogisticRegression(C=1,max_iter=1500,solver='liblinear',
        class_weight='balanced',random_state=RANDOM_STATE)
}
assert len(modelos_originales)==6
print('Modelos originales:',list(modelos_originales))


## 4.4. Cinco clasificadores adicionales

Los **cinco modelos de este apartado son nuevos respecto a la lista de seis modelos entrenados en la Parte A**. Registrarlos en `modelos_nuevos`, usando el mismo protocolo de evaluación de 4.1.4. No reescribir el split ni ajustar escaladores sobre todo el dataset. **Aldana** crea el diccionario en 4.4.1 y lo completa en los siguientes apartados.

### 4.4.1. SVM de clasificación (LinearSVC)

> **Responsable: Aldana**

**Implementar:** `LinearSVC` como variante lineal de SVM para controlar el costo computacional del dataset Santander; registrar con clave `svm_lineal`. Usar escalamiento e incrementar iteraciones solo si es necesario, explicando convergencia y margen.

**ROC-AUC:** `LinearSVC` no expone `predict_proba` de forma nativa; usar `decision_function` para ROC-AUC. No incluirlo directamente en híbridos de `voting='soft'` sin calibrar probabilidades.

**Entregable:** entrenamiento, métricas comparables e interpretación del comportamiento de SVM.

In [ ]:
# 4.4.1 — SVM lineal (Aldana)
modelos_nuevos={'svm_lineal': LinearSVC(C=1,dual=False,max_iter=10000,random_state=RANDOM_STATE)}
print('SVM registrada (decision_function para AUC)')


### 4.4.2. AdaBoostClassifier

> **Responsable: Aldana**

**Implementar:** `AdaBoostClassifier` como `adaboost`, con `random_state=42`; elegir un número razonable de estimadores y documentarlo.

**Verificación:** registrar Accuracy, Precision, Recall, F1 y ROC-AUC; comparar resultados con Decision Tree, su posible clasificador base.

**Entregable:** modelo registrable, métricas e interpretación.

In [ ]:
# 4.4.2 — AdaBoost (Aldana)
modelos_nuevos['adaboost'] = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1,random_state=RANDOM_STATE),
    n_estimators=60, learning_rate=.5,random_state=RANDOM_STATE)
print('AdaBoost registrado')


### 4.4.3. KNeighborsClassifier (KNN)

> **Responsable: Aldana**

**Implementar:** `KNeighborsClassifier` como `knn`. Elegir un valor inicial de `n_neighbors` y, si hay tiempo, compararlo mediante validación **solo sobre entrenamiento**.

**Precaución:** KNN es sensible a la escala, dimensión y costo de distancia; no contarlo como uno de los cuatro escaladores y evitar `KNNImputer` masivo salvo necesidad y justificación.

**Entregable:** modelo registrable y explicación de qué transformador le favoreció.

In [ ]:
# 4.4.3 — KNN (Aldana)
modelos_nuevos['knn'] = KNeighborsClassifier(n_neighbors=11,weights='distance',
    metric='minkowski',p=2,algorithm='brute',n_jobs=2)
print('KNN registrado')


### 4.4.4. LightGBMClassifier

> **Responsable: Aldana**

**Implementar:** `LGBMClassifier` como `lightgbm`, con semilla fija y parámetros moderados para Colab. Aunque la biblioteca aparezca en las **importaciones** originales, **no figura entre los seis clasificadores entrenados**, así que cuenta como modelo adicional.

**Verificación:** métricas sobre particiones acordadas y comparación con XGBoost/CatBoost.

**Entregable:** modelo registrado, configuración y evaluación.

In [ ]:
# 4.4.4 — LightGBM (Aldana)
modelos_nuevos['lightgbm'] = LGBMClassifier(objective='binary',n_estimators=90,
    learning_rate=.06,num_leaves=31,min_child_samples=20,
    colsample_bytree=.8,device_type='cpu',n_jobs=2,random_state=RANDOM_STATE,verbosity=-1)
print('LightGBM registrado')


### 4.4.5. ExtraTreesClassifier

> **Responsable: Aldana**

**Implementar:** `ExtraTreesClassifier` como `extra_trees`; fijar `random_state=42` y documentar número de árboles.

**Verificación:** comparar su comportamiento con Random Forest y explicar el mayor grado de aleatoriedad de Extra Trees.

**Entregable:** modelo registrado, métricas y comprobación final de `len(modelos_nuevos) == 5`.

In [ ]:
# 4.4.5 — Extra Trees (Aldana)
modelos_nuevos['extra_trees'] = ExtraTreesClassifier(n_estimators=90,
    min_samples_leaf=2,max_features='sqrt',n_jobs=2,random_state=RANDOM_STATE)
assert set(modelos_nuevos)=={'svm_lineal','adaboost','knn','lightgbm','extra_trees'}
print('Nuevos modelos:',list(modelos_nuevos))


## 4.5. Cinco modelos híbridos (ensambles)

Cada híbrido debe construirse con estimadores presentes en `modelos_originales` o `modelos_nuevos`. Registrar **H1–H5** en `modelos_hibridos`. En Voting Soft solo usar clasificadores que entregan probabilidades; en Stacking prevenir fugas con CV sobre entrenamiento. Todos se evalúan mediante la interfaz compartida. Los híbridos se registran por separado de los 11 modelos individuales.

### 4.5.1. H1 — VotingClassifier Hard: Logistic Regression + Decision Tree + KNN

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='hard')` con los tres estimadores indicados; registrar clave `H1_voting_hard`. Incorporar preprocesamiento adecuado para KNN/Logistic Regression dentro del procedimiento acordado.

**Explicar:** la etiqueta final se determina por voto mayoritario. **Precaución de evaluación:** en general hard voting no expone probabilidades; **no inventar ROC-AUC**. Si no existe puntuación apropiada, registrar `NaN` para ROC-AUC o evaluar una variante soft aparte documentada sin sustituir H1.

**Entregable:** híbrido entrenable, métricas válidas y breve explicación del mecanismo de voto.

In [ ]:
# 4.5.1 — H1 Voting Hard (Brickzon)
modelos_hibridos={}
modelos_hibridos['H1_voting_hard'] = VotingClassifier(estimators=[
    ('logistic_regression',Pipeline([('escalador',StandardScaler()),
        ('clasificador',clone(modelos_originales['logistic_regression']))])),
    ('decision_tree',clone(modelos_originales['decision_tree'])),
    ('knn',Pipeline([('escalador',StandardScaler()),
        ('clasificador',clone(modelos_nuevos['knn']))]))
],voting='hard',n_jobs=1)
print('H1: votación mayoritaria; ROC-AUC = NaN (no tiene score continuo)')


### 4.5.2. H2 — VotingClassifier Soft: Random Forest + AdaBoost + Logistic Regression

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='soft')` con los tres estimadores; registrar `H2_voting_soft`.

**Explicar:** combina probabilidades estimadas de clase 1. Comprobar que los tres modelos tengan `predict_proba`; usar sus puntuaciones para ROC-AUC de forma correcta. Escalamiento ajustado solo en entrenamiento.

**Entregable:** híbrido entrenable, métricas y comparación con cada componente.

In [ ]:
# 4.5.2 — H2 Voting Soft (Brickzon)
modelos_hibridos['H2_voting_soft']=VotingClassifier(estimators=[
    ('random_forest',clone(modelos_originales['random_forest'])),
    ('adaboost',clone(modelos_nuevos['adaboost'])),
    ('logistic_regression',Pipeline([('escalador',StandardScaler()),
        ('clasificador',clone(modelos_originales['logistic_regression']))]))
],voting='soft',n_jobs=1)
print('H2: promedio de probabilidades')


### 4.5.3. H3 — VotingClassifier Soft: XGBoost + LightGBM + CatBoost

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='soft')` con XGBoost, LightGBM y CatBoost; registrar `H3_voting_soft_boosting`.

**Precaución Colab:** usar versiones/argumentos compatibles, preferentemente CPU sin forzar GPU ni usar opciones obsoletas. Mantener semilla y criterios de evaluación comunes; usar estimadores clonables.

**Entregable:** híbrido entrenable, métricas y explicación de si combinar tres métodos boosting agrega valor respecto a cada uno.

In [ ]:
# 4.5.3 — H3 Voting Soft con tres algoritmos boosting (Brickzon)
modelos_hibridos['H3_voting_soft_boosting']=VotingClassifier(estimators=[
    ('xgboost',clone(modelos_originales['xgboost'])),
    ('lightgbm',clone(modelos_nuevos['lightgbm'])),
    ('catboost',clone(modelos_originales['catboost']))
],voting='soft',n_jobs=1)
print('H3: votación de XGBoost, LightGBM y CatBoost')


### 4.5.4. H4 — StackingClassifier: SVM + KNN + Random Forest

> **Responsable: Vera**

**Implementar:** `StackingClassifier` con `LinearSVC`, `KNeighborsClassifier` y `RandomForestClassifier` como modelos base; `LogisticRegression` como meta-modelo. Registrar `H4_stacking_svm`.

**Condiciones:** emplear `cv=3` (o justificar otro valor), `stack_method='auto'` y preprocesamiento compatible; construir pipelines internos para modelos sensibles a escala cuando corresponda. El meta-modelo se entrena con predicciones obtenidas **mediante CV del entrenamiento**; nunca con etiquetas del holdout.

**Entregable:** híbrido, reporte de costo de entrenamiento, métricas y explicación de las dos capas.

In [ ]:
# 4.5.4 — H4 Stacking: SVM + KNN + Random Forest (Vera)
cv_interna = StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
modelos_hibridos['H4_stacking_svm'] = StackingClassifier(estimators=[
    ('svm_lineal',crear_pipeline_santander(modelos_nuevos['svm_lineal'],StandardScaler())),
    ('knn',crear_pipeline_santander(modelos_nuevos['knn'],StandardScaler())),
    ('random_forest',crear_pipeline_santander(modelos_originales['random_forest']))
],final_estimator=LogisticRegression(max_iter=1500,random_state=RANDOM_STATE),
    cv=cv_interna,stack_method='auto',n_jobs=1,passthrough=False)
print('H4: las transformaciones de cada rama se ajustan dentro de CV interna')


### 4.5.5. H5 — StackingClassifier: Decision Tree + Extra Trees + AdaBoost

> **Responsable: Vera**

**Implementar:** `StackingClassifier` con `DecisionTreeClassifier`, `ExtraTreesClassifier` y `AdaBoostClassifier` como modelos base; `LogisticRegression` como meta-modelo. Registrar `H5_stacking_arboles`.

**Condiciones:** CV interna sobre entrenamiento, semillas consistentes, evitar remuestreo fuera de folds y no modificar datos de validación/holdout. Medir tiempo y observar sobreajuste.

**Entregable:** híbrido, métricas comparables, contraste con H4 y comprobación de `len(modelos_hibridos) == 5`.

In [ ]:
# 4.5.5 — H5 Stacking: Decision Tree + Extra Trees + AdaBoost (Vera)
modelos_hibridos['H5_stacking_arboles'] = StackingClassifier(estimators=[
    ('decision_tree',crear_pipeline_santander(modelos_originales['decision_tree'])),
    ('extra_trees',crear_pipeline_santander(modelos_nuevos['extra_trees'])),
    ('adaboost',crear_pipeline_santander(modelos_nuevos['adaboost']))
],final_estimator=LogisticRegression(max_iter=1500,random_state=RANDOM_STATE),
    cv=cv_interna,stack_method='auto',n_jobs=1,passthrough=False)
assert len(modelos_hibridos)==5
for est in modelos_hibridos.values(): clone(est)
print('Híbridos definidos:',list(modelos_hibridos))


## 4.6. Evaluación final, auditoría e integración

### 4.6.1. Consolidar resultados y seleccionar configuraciones mediante validación

> **Responsable: Marco**

**Implementar:** ejecutar los clasificadores `modelos_originales` y `modelos_nuevos` bajo los cuatro escaladores usando la misma estrategia de validación (idealmente CV estratificada en **entrenamiento**). Para los híbridos, seleccionar escalador/preprocesamiento usando la misma validación, registrando la decisión. Consolidar `resultados` en `tabla_resultados` con columnas homogéneas.

**Matriz sugerida:** hasta **4 × 11 = 44 configuraciones individuales** y **5 híbridos**. Esto es una **propuesta de evaluación**, no un número expresamente exigido por el profesor; si el costo de Colab obliga a priorizar, documentar cobertura real y justificarla. Si se usa una muestra exploratoria, las comparaciones deben usar la **misma muestra** y las conclusiones finales deben identificarse como tales.

**Selección:** priorizar ROC-AUC, Recall y F1 de clase 1; Accuracy no debe decidir sola por el desbalanceo. No elegir repetidamente configuraciones mirando el holdout.

**Entregable:** tabla completa, ordenamiento por métrica prioritaria y registro de configuraciones efectivamente probadas.

In [ ]:
# 4.6.1 — Evaluación única: 44 individuos x escalador + cinco híbridos (Marco)
COLUMNAS_CV=['tipo','modelo','escalador','cv_accuracy','cv_precision','cv_recall',
             'cv_f1','cv_roc_auc','cv_tiempo_s','folds','estado']
ESTADO_COMPARACION={'revision':0,'holdout_status':'sin_experimento'}

def ejecutar_comparacion():
    global tabla_validacion_ultima,tabla_resultados
    if ESTADO_COMPARACION['revision'] != 0:
        raise RuntimeError('La comparación CV solo se ejecuta una vez por sesión para proteger el holdout.')
    ESTADO_COMPARACION['revision']=1
    ESTADO_COMPARACION['holdout_status']='seleccionando'
    cv=StratifiedKFold(n_splits=CV_SPLITS,shuffle=True,random_state=RANDOM_STATE)
    filas_cv=[]
    registros=[]
    grupos=[('original',modelos_originales),('nuevo',modelos_nuevos),('hibrido',modelos_hibridos)]
    for tipo,dic in grupos:
        for nombre,est in dic.items():
            opciones= [('sin_escalado',None)] if tipo=='hibrido' else list(escaladores.items())
            for nombre_esc,esc in opciones:
                fold_rows=[]
                error=None
                for fold,(fi,vi) in enumerate(cv.split(X_train,y_train),start=1):
                    try:
                        fila=evaluar_modelo(nombre,tipo,est,esc,X_train.iloc[fi],y_train.iloc[fi],
                            X_train.iloc[vi],y_train.iloc[vi],fold=fold)
                        fila['escalador']=nombre_esc
                        fold_rows.append(fila)
                    except Exception as exc:
                        error=f'{type(exc).__name__}: {exc}'
                        break
                if error is not None:
                    filas_cv.append(dict(tipo=tipo,modelo=nombre,escalador=nombre_esc,
                        cv_accuracy=np.nan,cv_precision=np.nan,cv_recall=np.nan,
                        cv_f1=np.nan,cv_roc_auc=np.nan,cv_tiempo_s=np.nan,
                        folds=len(fold_rows),estado='ERROR: '+error))
                    print('ERROR',tipo,nombre,nombre_esc,':',error)
                    continue
                registros.extend(fold_rows)
                medias={f'cv_{m}':float(np.nanmean([r[m] for r in fold_rows]))
                        if np.any(np.isfinite([r[m] for r in fold_rows])) else np.nan
                        for m in ('accuracy','precision','recall','f1','roc_auc','tiempo_s')}
                filas_cv.append(dict(tipo=tipo,modelo=nombre,escalador=nombre_esc,
                    folds=len(fold_rows),estado='OK',**medias))
                print(f'CV {len(filas_cv):02d}/49 · {tipo}/{nombre}/{nombre_esc} OK')
    tabla_validacion_ultima=pd.DataFrame(filas_cv,columns=COLUMNAS_CV)
    tabla_resultados=pd.DataFrame(registros,columns=COLUMNAS_RESULTADOS)
    ESTADO_COMPARACION['holdout_status']='cv_disponible'
    if len(tabla_validacion_ultima)!=49:
        raise AssertionError('Deben existir exactamente 44 combinaciones individuales y 5 híbridos.')
    if STRICT_ERRORS and not tabla_validacion_ultima['estado'].eq('OK').all():
        fallas=tabla_validacion_ultima[~tabla_validacion_ultima['estado'].eq('OK')]
        display(fallas)
        raise RuntimeError('La CV no terminó correctamente. No se evaluará el holdout.')
    return tabla_validacion_ultima

def evaluar_holdout_final():
    global tabla_resultados
    if ESTADO_COMPARACION['holdout_status']!='cv_disponible':
        raise RuntimeError('Debe existir una CV completa y no haber evaluado holdout previamente.')
    cv=tabla_validacion_ultima.copy()
    correctas=cv[cv['estado'].eq('OK')].copy()
    if correctas.empty:
        raise RuntimeError('Sin modelos válidos para seleccionar.')
    correctas['_ranking_auc']=correctas['cv_roc_auc'].fillna(-np.inf)
    seleccion=correctas.sort_values(['_ranking_auc','cv_recall','cv_f1'],ascending=False)
    seleccion=seleccion.drop_duplicates(['tipo','modelo'],keep='first')
    ESTADO_COMPARACION['holdout_status']='evaluando'
    filas=[]
    try:
        for _,item in seleccion.iterrows():
            tipo,nombre,nombre_esc=item['tipo'],item['modelo'],item['escalador']
            estimador=dict(original=modelos_originales,nuevo=modelos_nuevos,
                           hibrido=modelos_hibridos)[tipo][nombre]
            escalador=escaladores[nombre_esc] if nombre_esc!='sin_escalado' else None
            fila=evaluar_modelo(nombre,tipo,estimador,escalador,X_train,y_train,
                 X_test,y_test,etapa='holdout',guardar_ajustado=True)
            fila['escalador']=nombre_esc
            filas.append(fila)
            print('HOLDOUT ·',tipo,nombre,nombre_esc,'OK')
        tabla_resultados=pd.concat([tabla_resultados,pd.DataFrame(filas)],ignore_index=True)
        ESTADO_COMPARACION['holdout_status']='evaluado'
    except Exception:
        ESTADO_COMPARACION['holdout_status']='fallido'
        raise
    return tabla_resultados.query("etapa == 'holdout'").copy()


### 4.6.2. Verificar los cinco híbridos y sus métricas

> **Responsable: Vera**

**Implementar:** comprobar que `modelos_hibridos` contiene H1–H5 y que sus componentes están presentes entre los modelos individuales. Evaluar los cinco con la interfaz común y un preprocesamiento comparable.

**Auditoría mínima:** comparar H1/H2/H3 frente a sus componentes; H4/H5 frente a sus modelos base; revisar `ROC-AUC` con etiquetas reales y puntuaciones (para hard voting, `NaN` si no hay puntuación válida); registrar tiempos y errores/convergencia sin ocultarlos.

**Entregable:** tabla `tabla_hibridos`, con cinco filas por la configuración final elegida y una nota sobre rendimiento/costo.

In [ ]:
# 4.6.2 — Auditoría de solo lectura sobre los híbridos (Vera)
def evaluar_y_auditar_hibridos_vera():
    nombres=['H1_voting_hard','H2_voting_soft','H3_voting_soft_boosting',
             'H4_stacking_svm','H5_stacking_arboles']
    if set(nombres)!=set(modelos_hibridos):
        raise AssertionError('Se requieren exactamente H1–H5.')
    if tabla_validacion_ultima.empty:
        print('Evaluación híbrida pendiente: primero se requiere la CV de Marco.')
        return pd.DataFrame()
    cobertura=tabla_validacion_ultima.query("tipo == 'hibrido'")
    if set(cobertura['modelo'])!=set(nombres) or not cobertura['estado'].eq('OK').all():
        raise AssertionError('CV de los cinco híbridos ausente o con errores.')
    finales=tabla_resultados.query("tipo == 'hibrido' and etapa == 'holdout'").copy()
    if len(finales)!=5 or not finales['modelo'].is_unique:
        raise AssertionError('Se necesitan cinco evaluaciones finales independientes.')
    h1=finales.query("modelo == 'H1_voting_hard'")
    if not h1['roc_auc'].isna().all():
        raise AssertionError('H1 hard voting no debe inventar ROC-AUC continuo.')
    for c in ('accuracy','precision','recall','f1'):
        if not finales[c].between(0,1).all(): raise AssertionError('Métricas fuera de rango: '+c)
    print('Auditoría de los cinco híbridos: OK; sin modificaciones ni entrenamiento adicional.')
    return finales


### 4.6.3. Generar visualizaciones y matrices de confusión

> **Responsable: Marco**

**Implementar:** tabla ordenada de rendimiento individual e híbrido, gráficos legibles (por ejemplo, barras de ROC-AUC/F1/Recall) y matrices de confusión de los modelos más relevantes usando el **holdout final**. Si se trazan curvas ROC, usar predicciones probabilísticas o puntuaciones sobre `X_test` y `y_test`, sin invertir los argumentos.

**Entregable:** gráficos con etiquetas y títulos, interpretación de falsos negativos (clientes insatisfechos no detectados) y advertencia del desbalanceo.

**Verificación:** las etiquetas y los cálculos provienen de la misma partición, no mezclan validación con holdout.

In [ ]:
# 4.6.3 — Tablas y visualizaciones (Marco)
def mostrar_tabla_resultados():
    cv=tabla_validacion_ultima.sort_values(['cv_roc_auc','cv_recall'],ascending=False)
    print('Comparación por validación cruzada (selección, no holdout):')
    display(cv[['tipo','modelo','escalador','cv_roc_auc','cv_recall','cv_f1','cv_tiempo_s','estado']])
    print('Evaluación holdout, una configuración por modelo:')
    finales=tabla_resultados.query("etapa == 'holdout'").copy()
    display(finales[['tipo','modelo','escalador','accuracy','precision','recall','f1','roc_auc','tiempo_s']])
    return finales

def graficar_metricas():
    finales=tabla_resultados.query("etapa == 'holdout'").copy()
    if finales.empty: return
    finales['config']=finales['modelo']+' ('+finales['escalador']+')'
    for metrica in ['roc_auc','recall','f1']:
        fig, ax=plt.subplots(figsize=(13,6))
        vista=finales.sort_values(metrica,ascending=False)
        sns.barplot(data=vista,x='config',y=metrica,hue='tipo',ax=ax,dodge=False)
        ax.tick_params(axis='x',rotation=80)
        ax.set(title=f'Holdout — {metrica}',xlabel='Configuración')
        plt.tight_layout();plt.show()

def graficar_matrices_confusion():
    finales=tabla_resultados.query("etapa == 'holdout'")
    for _,fila in finales.iterrows():
        if fila['matriz_confusion'] is None: continue
        fig,ax=plt.subplots(figsize=(4,3.5))
        sns.heatmap(np.asarray(fila['matriz_confusion']),annot=True,fmt='d',cmap='Blues',ax=ax,
            xticklabels=[0,1],yticklabels=[0,1],cbar=False)
        ax.set(xlabel='Predicción',ylabel='TARGET real',title=fila['modelo'])
        plt.tight_layout();plt.show()

def graficar_curvas_roc():
    fig,ax=plt.subplots(figsize=(8,6))
    count=0
    for (tipo,nombre),p in predicciones_holdout.items():
        if p['score'] is None:continue
        fpr,tpr,_=roc_curve(p['y_true'],p['score'])
        ax.plot(fpr,tpr,label=f'{nombre} ({roc_auc_score(p["y_true"],p["score"]):.3f})')
        count+=1
    if count==0:plt.close(fig);return
    ax.plot([0,1],[0,1],linestyle='--',color='gray')
    ax.set(xlabel='FPR',ylabel='TPR',title='ROC — holdout final')
    ax.legend(fontsize='x-small',loc='lower right',ncol=2)
    plt.tight_layout();plt.show()


### 4.6.4. Redactar interpretación, conclusiones y limitaciones

> **Responsable: Marco**

**Implementar en celdas de texto Markdown debajo de este bloque:** justificar cuál escalador funcionó mejor y para qué modelos; comparar individuales versus híbridos; explicar qué métricas sostienen la selección; comentar tiempo computacional, desbalanceo y límites de la evaluación.

**No inventar resultados.** Completar las conclusiones únicamente después de ejecutar las celdas y revisar la tabla consolidada. Indicar si alguna combinación se omitió y el motivo.

**Entregable:** un párrafo de resultados, un párrafo comparativo y un párrafo de limitaciones y mejoras. La celda de código vacía se deja como espacio opcional para cálculos auxiliares de este análisis.

In [ ]:
# 4.6.4 — Resumen reproducible para conclusiones (Marco)
def preparar_resumen_conclusiones():
    cv=tabla_validacion_ultima.copy()
    cv['_auc_rank']=cv['cv_roc_auc'].fillna(-np.inf)
    ganadores=cv.sort_values(['_auc_rank','cv_recall','cv_f1'],ascending=False).drop_duplicates(['tipo','modelo'])
    print('Configuraciones seleccionadas mediante CV:')
    display(ganadores[['tipo','modelo','escalador','cv_roc_auc','cv_recall','cv_f1']])
    final=tabla_resultados.query("etapa == 'holdout'").copy()
    if final.empty:
        print('Holdout todavía no realizado. No hay conclusiones experimentales definitivas.')
        return ganadores
    print('Interpretar especialmente recall y F1 de clientes insatisfechos (TARGET=1).')
    print('Limitación metodológica:',
          'entrenamiento acotado a '+str(MAX_FILAS_ENTRENAMIENTO)+' registros' if MAX_FILAS_ENTRENAMIENTO else
          'entrenamiento con todas las filas disponibles.')
    print('No comparar tiempos de Colab como benchmarks de hardware estandarizado.')
    return ganadores


### 4.6.5. Control final de integración y ejecución en Google Colab

> **Responsable: Marco**

**Implementar:** comprobaciones automáticas/focalizadas que verifiquen: 4 escaladores, 6 modelos originales, 5 modelos nuevos, 5 híbridos; forma y clases de `X_train/X_test/y_train/y_test`; inexistencia de `ID` entre predictores; ausencia de fuga antes de split; nombres únicos de modelos y métricas válidas. Verificar que `tabla_resultados` no esté vacía y no contenga resultados inventados.

**Entrega:** guardar el `.ipynb` final con Markdown completo, celdas de ejecución y resultados reales, más una copia descargable desde Colab. Si se utilizó Drive, eliminar dependencias de rutas privadas o documentar cómo reconstruirlas. Ejecutar desde una sesión limpia **la Parte B en orden**, sin depender de variables definidas en la Parte A.

**Criterio de aceptación:** cualquier integrante puede abrir el notebook, ubicar los datos, seguir las instrucciones, ejecutar el bloque grupal y entender quién hizo cada parte.

In [ ]:
# 4.6.5 — EJECUTAR TODO: pruebas de contrato, 49 configuraciones, holdout y gráficos
# Esta es la única celda que inicia entrenamientos y publicación de resultados.
assert set(escaladores)=={'standard','minmax','robust','quantile'}
assert len(modelos_originales)==6 and len(modelos_nuevos)==5 and len(modelos_hibridos)==5
for d in (modelos_originales,modelos_nuevos,modelos_hibridos):
    for estimador in d.values():
        clone(estimador)
assert y_train.index.equals(X_train.index) and y_test.index.equals(X_test.index)
assert not X_train.index.intersection(X_test.index).size
print('Contrato de código: OK. Comienza CV de 44 configuraciones individuales + 5 híbridos.')
tabla_cv=ejecutar_comparacion()
tabla_holdout=evaluar_holdout_final()
tabla_hibridos=evaluar_y_auditar_hibridos_vera()
mostrar_tabla_resultados()
graficar_metricas()
graficar_matrices_confusion()
graficar_curvas_roc()
preparar_resumen_conclusiones()
print('EJECUCIÓN INTEGRAL TERMINADA · CV:',len(tabla_cv),'configuraciones; holdout:',len(tabla_holdout),'modelos.')


## 4.7. Comprobaciones finales y entrega

Al terminar «Ejecutar todo»:

- Verificar 49 filas en `tabla_validacion_ultima` y 16 evaluaciones finales en `tabla_holdout` (11 individuales y 5 híbridos).
- Verificar 5 filas de `tabla_hibridos`, métricas reales, matrices de confusión y curvas ROC para los modelos que producen puntajes continuos.
- **Modo completo:** se utiliza el conjunto de entrenamiento disponible después del split. Si se cambia el parámetro a 8 000, los resultados deberán indicar explícitamente que se entrenó con una muestra estratificada.
- Descargar el notebook desde Colab con sus salidas y registrar la configuración, versiones y limitaciones.
- **Nunca recalcular CV para elegir nuevamente un modelo usando información del holdout**.

### Checklist de equipo y cierre

- [ ] **Poma:** split sin fugas y 4 escaladores ajustados solo con entrenamiento.
- [ ] **Aldana:** 5 modelos nuevos registrados.
- [ ] **Brickzon:** H1/H2/H3 con votación y probabilidades válidas donde corresponda.
- [ ] **Vera:** H4/H5 con transformación dentro de CV interna y auditoría de solo lectura.
- [ ] **Marco:** 6 modelos base, CV 44+5, único holdout, tablas y conclusiones.
- [ ] **Equipo:** se completó la ejecución real de Colab y se registraron límites de muestra y entorno.

Esta versión está preparada para ejecutar todo el flujo sin código pendiente; **los resultados científicos se generan exclusivamente al ejecutar el notebook con el CSV real**.